In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from src.data_utils import preprocess_dataset, split_dataset
import os
import urllib.request

url = "https://code.s3.yandex.net/deep-learning/tweets.txt"
# создаем папку data если нет


raw_path = "data/raw_dataset.csv"
processed_path = "data/dataset_processed.csv"
data_dir = "data"

os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(raw_path):    
    with urllib.request.urlopen(url) as response:
        lines = response.read().decode("utf-8").splitlines()

    df = pd.DataFrame(lines, columns=["text"])
    df.to_csv(raw_path, index=False)

# Очистка
if not os.path.exists(processed_path):
    preprocess_dataset(raw_path, processed_path)

# Разбиение
if not os.path.exists("data/train.csv"):
    split_dataset(processed_path, data_dir)

In [ ]:
tokenizer_lstm = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
from torch.utils.data import DataLoader
from src.next_token_dataset import NextTokenDataset, collate_fn

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

train_df = train_df.dropna()[:10000]
val_df = val_df.dropna()
test_df = test_df.dropna()

train_ids = tokenizer_lstm(train_df["text"].tolist(), add_special_tokens=False)["input_ids"]
val_ids = tokenizer_lstm(val_df["text"].tolist(), add_special_tokens=False)["input_ids"]
test_ids = tokenizer_lstm(test_df["text"].tolist(), add_special_tokens=False)["input_ids"]

train_dataset = NextTokenDataset(train_ids)
val_dataset = NextTokenDataset(val_ids)
test_dataset = NextTokenDataset(test_ids)

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

In [ ]:
import torch.nn as nn
from src.lstm_model import LSTMModel

lstm_model = LSTMModel(
    vocab_size=tokenizer_lstm.vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=1,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer_lstm.pad_token_id)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.002)

In [ ]:
from src.lstm_train import train
n_epoch = 2
train(lstm_model, n_epoch, criterion, optimizer, tokenizer_lstm, train_loader, val_loader, device)
torch.save(lstm_model.state_dict(), "models/lstm_model.pth")

In [ ]:
import pandas as pd
from src.eval_transformer_pipeline import (
    build_transformer,
    evaluate_transformer,
    show_example_models
)
# создаем transformer
generator, tokenizer = build_transformer()

# считаем ROUGE
rouge1, rouge2 = evaluate_transformer(generator, tokenizer, val_df)

print(f"Transformer ROUGE-1: {rouge1:.4f} | ROUGE-2: {rouge2:.4f}")

# показываем примеры
show_example_models(
    generator,
    tokenizer,
    test_df,
    lstm_model,
    tokenizer_lstm,
    num_examples=3
)

Выводы:
